# Experiment 4 Analysis: VC Repair Algorithms Comparison

This notebook analyzes the performance of three VC repair algorithms:
1. **Vanilla VC**: Maximal degree vertex removal.
2. **Classic VC**: Random edge removal.
3. **Weighted VC**: Dynamic alpha-based removal.

The analysis is conducted across different epsilon values (privacy budgets) and datasets (Adult, Census, Compas, Tax).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import os

sns.set_theme(style="whitegrid")

# Load data
df = pd.read_csv('remote/output/experiment_4_repair_comparison_labeled.csv')

def safe_parse(val):
    if pd.isna(val): return {}
    try:
        # Handle both string dicts and actual dicts if any
        if isinstance(val, str):
            # Some might use single quotes, some double. ast.literal_eval is safer than json.loads for single quotes.
            return ast.literal_eval(val)
        return val
    except:
        return {}

# Preprocess nested columns
print("Parsing nested columns...")
df['del_ratio_val'] = df['deletion_ratio'].apply(lambda x: safe_parse(x).get('ratio', 0))
df['err_syn'] = df['marginals_error'].apply(lambda x: safe_parse(x).get('synthetic_avg', 0))
df['err_rep'] = df['marginals_error'].apply(lambda x: safe_parse(x).get('repaired_avg', 0))
df['tvd_syn'] = df['tvd_2way'].apply(lambda x: safe_parse(x).get('synthetic_avg', 0))
df['tvd_rep'] = df['tvd_2way'].apply(lambda x: safe_parse(x).get('repaired_avg', 0))

# ML Accuracy (average across models)
def get_avg_ml(val, key):
    d = safe_parse(val).get(key, {})
    if not d: return 0
    return np.mean(list(d.values()))

df['ml_syn'] = df['ml_accuracy'].apply(lambda x: get_avg_ml(x, 'synthetic'))
df['ml_rep'] = df['ml_accuracy'].apply(lambda x: get_avg_ml(x, 'repaired'))

print(f"Loaded {len(df)} records.")
df.head()

## 1. Deletion Ratio Comparison
Lower deletion ratio is generally better as it preserves more data points.

In [ ]:
g = sns.relplot(
    data=df,
    x="epsilon", y="del_ratio_val",
    hue="repair_algorithm", col="dataset",
    kind="line", marker="o",
    facet_kws={'sharey': False}
)
g.set_axis_labels("Epsilon (Privacy Budget)", "Deletion Ratio")
g.fig.suptitle("Deletion Ratio vs Epsilon by Repair Algorithm", y=1.02)
plt.show()

## 2. Marginal Error Utility
Comparison of average marginal error between repair algorithms. We want to see how much error increases due to repair.

In [ ]:
g = sns.relplot(
    data=df,
    x="epsilon", y="err_rep",
    hue="repair_algorithm", col="dataset",
    kind="line", marker="s",
    facet_kws={'sharey': False}
)
g.set_axis_labels("Epsilon (Privacy Budget)", "Repaired Average Marginal Error")
g.fig.suptitle("Utility (Marginal Error) vs Epsilon", y=1.02)
plt.show()

## 3. ML Accuracy Impact
Average accuracy across Logistic Regression, Random Forest, and MLP.

In [ ]:
g = sns.relplot(
    data=df,
    x="epsilon", y="ml_rep",
    hue="repair_algorithm", col="dataset",
    kind="line", marker="^",
    facet_kws={'sharey': False}
)
g.set_axis_labels("Epsilon (Privacy Budget)", "Repaired ML Accuracy (Avg)")
g.fig.suptitle("Utility (ML Accuracy) vs Epsilon", y=1.02)
plt.show()

## 4. Synthesizer Comparison (AIM vs MST)
Let's see if the repair behavior differs significantly between synthetically generated data from AIM vs MST.

In [ ]:
g = sns.relplot(
    data=df,
    x="epsilon", y="del_ratio_val",
    hue="repair_algorithm", row="synthesizer", col="dataset",
    kind="line", marker="o",
    height=3, aspect=1.2,
    facet_kws={'sharey': False}
)
plt.show()

## Summary of Findings
- **Vanilla VC** vs **Classic VC**: Compare the impact of greedy removal (degree) vs random removal.
- **Weighted VC**: Analyze if the dynamic alpha improves utility preservation while ensuring zero violations.
- **Epsilon Trend**: Generally, higher epsilon (less noise) should result in lower deletion ratios as the synthetic data is already "closer" to the constraints (if the private data satisfies them).